# Task 1 - Classification
The task is to develop a multi-task deep learning model inorder to predict artist, style, and genre of paintings from the WikiArt dataset.

My solution uses a CNN Backbone (EfficentNet b3 to be specific) for visual feature extraction along with RNN (BiLSTM to be specific) layers to learn the spatial relationships between image regions.

Finally, I've developed an outlier detection logic inorder to detect outliers i.e paintings that do not fit a particular artist or genre despite their assignment.

## Introduction

Inorder to build a model that can successfully classify three tasks (artist,style and genre) from a shared feature representation, the model was designed using the following properties:

• EfficientNet-B3 (Noisy Student) was chosen as the CNN backbone to extract visual features from the painting.
Why?:
- Best tradeoff between accuracy and compute required (12M parameters and 300 size i/p).
- EfficientNet performs better for fine-grained visual classification problems. EfficientNet’s architecture (MBConv blocks + squeeze-excitation) captures fine texture patterns very well.

• A bidirectional LSTM was chosen as the RNN for modelling spatial relationships between different regions of the image.
Why?:
- Standard RNNs suffer from vanishing gradients
- A standard RNN can process the sequence only in one direction but spatial patches in an image have no natural order.

• Outlier detection uses confidence agreement + k-Nearest Neighbour logic to detect top 2% outliers in each group of 20 images.

## Dataset
The WikiArt dataset was used here as per requirements.

The original WikiArt dataset contains about 80k images. But for my problem definition i.e classifying on 3 heads (artist, style, genre), it is required that each image in the dataset has all 3 labels. But this was not the case.

Inorder to overcome this challenge, I took the inner join of all three labels and created a subset of WikiArt containing ~16k images (11.2k training + 4.7k validation split)

The official csv files are structured in the following format : imagepath, artist, genre, style.
It also provides text files containing all labels of each class. This text file was used to create a mapping between class indexes and label names in several parts of the program.

Another challenge was that when I performed the inner join 11 classes out of the 27 genre classes ended up having 0 classes. Also diagnostics revealed a 772x imbalance in genre and 347x imbalance in style. I failed to notice these details initially and obtained ~1% accuracy for style and genre.

The missing classes were removed from and the labels were made continous from 0-15. Inorder to solve the imbalance issue, I switched Cross Entropy Loss with Focal Loss which I will discuss about later in this notebook.








## Codes


Mount Google drive to upload the filtered WikiArt dataset and csv file. It also provides the checkpoints and result storage location.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Copy the dataset and csv file to colab env for faster processing.

In [ ]:
!cp -r /content/drive/MyDrive/wikiart_filtered /content/
!cp -r /content/drive/MyDrive/wikiart_csv /content/

These are paths required for the program to run properly.

In [ ]:
root_dir = "/content/wikiart_filtered"
train_csv = "/content/wikiart_csv/train_labels_fixed.csv"
val_csv = "/content/wikiart_csv/val_labels_fixed.csv"
artist_map = "/content/wikiart_csv/artist_class.txt"
genre_map = "/content/wikiart_csv/genre_class.txt"
style_map = "/content/wikiart_csv/style_class.txt"
csv_path = "/content/wikiart_csv/train_labels_fixed.csv"
checkpoint_dir="/content/drive/MyDrive/artextract_checkpoints"
results_dir="/content/drive/MyDrive/artextract_results"

### Imports

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.amp import autocast,GradScaler
import matplotlib.pyplot as plt
import os
from sklearn.metrics import f1_score
import torch.nn.functional as F
import numpy as np
import pandas as pd
from sklearn.neighbors import NearestNeighbors
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader,WeightedRandomSampler
from torchvision.transforms import v2
from PIL import Image
import timm

### Dataset

Each line in the file contains the name of a class (artist, genre, or style).
The function reads the file line-by-line and creates a dictionary that maps class index to class name.

In [ ]:
def load_mapping(filepath):
    mapping={}
    with open(filepath, "r") as f:
        for idx, line in enumerate(f):
            mapping[idx] = line.strip()
    return mapping

WikiArtSupervisedDataset is the dataset class used to load images and their labels for training and evaluation.

- It reads image paths and labels from a CSV file

- Loads the corresponding painting image from disk

- Applies image transformations

- Each sample returned by the dataset has the format: (image, style_label, genre_label, artist_label)

In [ ]:
class WikiArtSupervisedDataset(Dataset):
    def __init__(self,root_dir,csv_file
    ,artist_map,genre_map,style_map,transform=None):

        self.root_dir=root_dir
        self.transform=transform

        self.data=pd.read_csv(csv_file,header=None)

        self.image_path = self.data[0].tolist()
        self.artist_labels = self.data[1].astype(int).tolist()
        self.genre_labels = self.data[2].astype(int).tolist()
        self.style_labels = self.data[3].astype(int).tolist()

        self.idx_to_artist = load_mapping(artist_map)
        self.idx_to_genre = load_mapping(genre_map)
        self.idx_to_style = load_mapping(style_map)

    def __len__(self):
        return len(self.image_path)



    def __getitem__(self, ids):
        img_path=os.path.join(self.root_dir, self.image_path[ids])

        try:
            with Image.open(img_path) as img:
                img = img.convert("RGB")
        except Exception as e:
            print(f"Error loading image {img_path}: {e}")
            img = Image.new('RGB', (300,300), (0, 0, 0))

        if self.transform:
            img=self.transform(img)

        style=torch.tensor(self.style_labels[ids], dtype=torch.long)
        genre=torch.tensor(self.genre_labels[ids], dtype=torch.long)
        artist=torch.tensor(self.artist_labels[ids], dtype=torch.long)
        return img,style,genre,artist

### DataLoader

I have defined the getdataloader() fn to create the training and validation data loaders for the WikiArt dataset, including augmentation and class imbalance handling.

1. Defines transforms to be applied for training and validation datasets

2. Creates train and val dataset objects using the transforms defined

3. Computes class frequencies

4. Uses WeightedRandomSampler to oversample rare classes.

5. Returns training and validation dataloaders

Why WeightedRandomSampler is used here:
The dataset is strongly imbalanced(347x for style,772x for genre). Normal sampling would bias the model toward dominant classes (e.g., Impressionism or landscape). Weighted sampling increases the probability of training on rare styles and genres.

In [ ]:
def getdataloader(train_csv,val_csv, root_dir,artist_map,genre_map,style_map):

    # augmentations performed: EfficientNet b3(my CNN backbone) uses 300x300 inputs so we do RandomResizedCrop(300), then images are randomly flipped horizontally with a probability of 0.5
    # we only apply mild colorjitter to preserve the original colours and textures of the painting
    # the image is converted to a tensor and normalised using mean and standard deviation of Imagenet.
    train_transform=v2.Compose([v2.RandomResizedCrop(300),
                v2.RandomHorizontalFlip(p=0.5),
                v2.ColorJitter(brightness=0.1,contrast=0.1,saturation=0.1,hue=0.02),
                v2.ToTensor(),
                v2.Normalize(mean=[0.485, 0.456, 0.406],std=[0.229, 0.224, 0.225])])

    val_transform=v2.Compose([v2.Resize(300),v2.CenterCrop(300),
                              v2.ToTensor(),
                                v2.Normalize(mean=[0.485, 0.456, 0.406],
                                             std=[0.229, 0.224, 0.225])]) #validatino transforms are deterministic (no randomness)


    train_dataset = WikiArtSupervisedDataset(
        root_dir,
        train_csv,
        artist_map,
        genre_map,
        style_map,
        transform=train_transform
    )

    val_dataset = WikiArtSupervisedDataset(
        root_dir,
        val_csv,
        artist_map,
        genre_map,
        style_map,
        transform=val_transform
    )

    print(f"No of training samples:",len(train_dataset))
    print(f"No of validation samples:",len(val_dataset))

    #extract labels to compute sample weights
    genre_labels = torch.tensor(train_dataset.genre_labels)
    style_labels = torch.tensor(train_dataset.style_labels)

    #count number of samples in each class
    style_counts = torch.bincount(torch.tensor(style_labels))
    genre_counts = torch.bincount(torch.tensor(genre_labels))

    #reciprocal is taken to assign larger weights to rarer classes
    #clamp to avoid div by 0
    style_weights=1.0/style_counts.clamp(min=1).float()
    genre_weights=1.0/genre_counts.clamp(min=1).float()


    sample_weights = style_weights[style_labels]+ genre_weights[genre_labels]

    sampler=WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)
    train_loader=DataLoader(train_dataset,batch_size=8,sampler=sampler,num_workers=2,pin_memory=True,persistent_workers=True)
    val_loader=DataLoader(val_dataset,batch_size=8,shuffle=False,num_workers=2,pin_memory=True,persistent_workers=True)
    return train_loader, val_loader

The compute_weights() fn computes class weights for a given label set.

In [ ]:
def compute_weight(labels):
    labels=torch.tensor(labels)
    class_counts = torch.bincount(labels)
    num_classes = len(class_counts)
    total_samples = len(labels)

    weights=total_samples / (num_classes * class_counts.clamp(min=1).float())
    return weights

The weight_values() fn computes class weights for all three tasks.
The class weights are used inorder to penalize the rare class mistakes more.

It loads the dataset, extracts labels and then calls compute_weight() for each label type

In [ ]:
def weight_values(csv_path, root_dir, artist_map, genre_map, style_map):
    dataset= WikiArtSupervisedDataset(root_dir,csv_path,artist_map,genre_map,style_map,transform=None)
    style_weights=compute_weight(dataset.style_labels)
    genre_weights=compute_weight(dataset.genre_labels)
    artist_weights=compute_weight(dataset.artist_labels)
    return style_weights, genre_weights, artist_weights

### Model Architecture

The network uses EfficientNet-B3 (Noisy Student) from the timm library as a CNN backbone to extract features from the input image. Instead of producing classification outputs, the backbone outputs a feature map representing spatial information across the image.

The feature map has shape: (batch_size, channels, height, width)

The feature map is then reshaped into a sequence of patches so that it can be input to the BiLSTM: (batch_size, height × width, channels)

The outputs of the BiLSTM are then mean-pooled across the sequence dimension to produce a 1024-dimensional feature representation of the entire painting. A dropout layer is applied for regularization.

The shared feature vector is passed through three separate fc layers (classification heads) that predicts artists, genre, and style.

The model can also optionally return the intermediate feature embedding which is later used for outlier detection pipeline.

In [ ]:
class CNN_BiLSTM(torch.nn.Module):
    def __init__(self,num_style,num_genre,num_artists):
        super().__init__()

        #first the cnn backbone is defined. I have picked the efficientnet b3 noisy student variant here.
        self.cnn_backbone=timm.create_model('tf_efficientnet_b3_ns', pretrained=True, num_classes=0, global_pool="")

        self.feature_dim=self.cnn_backbone.num_features

        #now we define the biLSTM
        self.bilstm=torch.nn.LSTM(
            input_size=self.feature_dim,
            hidden_size=512,
            num_layers=2,
            batch_first=True,
            bidirectional=True
        )

        #since the bilstm is bidirectional,o/p dimension will be 512*2=1024
        bilstm_out_dim=1024

        self.dropout=torch.nn.Dropout(0.3)

        #inorder to get three labels as outputs,we design three fc layers for final label generations.
        self.style_head=torch.nn.Linear(bilstm_out_dim,num_style)
        self.genre_head=torch.nn.Linear(bilstm_out_dim,num_genre)
        self.artist_head=torch.nn.Linear(bilstm_out_dim,num_artists)

    def forward(self,x,return_features=False):
        cnn_features=self.cnn_backbone.forward_features(x)
        B,C,H,W=cnn_features.shape

        #cnn_features has shape (batch_size,channels,feature_map_height,feature_map_width)
        #but lstm expects input of shape (batch_size,sequence_len,feature_dim)
        #so we reshape to (batch_size,height*width,channels)
        cnn_features=cnn_features.reshape(B,C,H*W).permute(0,2,1)

        #ignore hidden states.
        lstm_out,_=self.bilstm(cnn_features)

        #use mean pooling to get summary of all patches.
        fc_in = lstm_out.mean(dim=1)
        fc_in=self.dropout(fc_in)

        style_out=self.style_head(fc_in)
        genre_out=self.genre_head(fc_in)
        artist_out=self.artist_head(fc_in)

        if return_features:
            return style_out, genre_out, artist_out, fc_in
        return style_out, genre_out, artist_out

### Loss functions
We use standard CrossEntropyLoss for artist classification but due to severe class imbalance in style and genre data, We use FocalLoss to compute their losses.

In highly imbalanced datasets, standard Cross-Entropy Loss tends to be dominated by majority classes. Easy examples (which the model already predicts correctly with high confidence) produce large numbers of low-loss contributions that overwhelm the learning signal from minority classes.

Focal Loss modifies cross-entropy so that:

- Well-classified samples contribute less to the loss

- Hard samples contribute more to the loss.

This forces the model to focus learning on rare cases.

---

**Mathematical Formulation**

Standard cross-entropy loss : $CE(p_t) = -\log(p_t)$

Focal Loss : $FL(p_t) = -\alpha (1 - p_t)^\gamma \log(p_t)$

Where:
$p_t$ = probability of correct class

$\gamma$ = focusing parameter

$(1-p_t)^\gamma$ reduces loss for easy samples



In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.ce = nn.CrossEntropyLoss(weight=alpha, reduction="none")

    def forward(self, inputs, targets):
        ce_loss = self.ce(inputs, targets)
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        return focal_loss.mean()

### Training

Device Setup

Select T4 GPU runtime from Runtime menu

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(f"Using device:{device}")
os.makedirs("/content/checkpoints", exist_ok=True)
os.makedirs("/content/results", exist_ok=True)

Use the getdataloader() method to get the data loader for training and validation.

In [ ]:
train_loader,val_loader=getdataloader(train_csv=train_csv,val_csv=val_csv,root_dir=root_dir,artist_map=artist_map,genre_map=genre_map,style_map=style_map)


Using the weight_values() method we previously defined we can get the class weights which are used in loss fns.

In [ ]:
style_weights, genre_weights, artist_weights= (weight_values(csv_path=csv_path, root_dir=root_dir, artist_map=artist_map, genre_map=genre_map, style_map=style_map))
style_weights=style_weights.to(device)
genre_weights=genre_weights.to(device)
artist_weights=artist_weights.to(device)
num_style = len(style_weights)
num_genre = len(genre_weights)
num_artists = len(artist_weights)

Model instantiation

In [ ]:
model=CNN_BiLSTM(num_style=num_style,num_genre=num_genre,num_artists=num_artists).to(device)

for p in model.cnn_backbone.parameters():
    p.requires_grad = False


Define the loss fns.

We use CrossEntropyLoss for artist and FocalLoss for style and genre.



In [ ]:
style_loss_fn = FocalLoss(alpha=style_weights,gamma=1.5)
genre_loss_fn = FocalLoss(alpha=genre_weights, gamma=2)
artist_loss_fn=nn.CrossEntropyLoss(weight=artist_weights,label_smoothing=0.1)

Now we define the optimiser, no of epochs, scaler and cosine learning rate scheduler that we will use for training the model.

In [ ]:
optimizer=optim.AdamW(model.parameters(),lr=3e-4,weight_decay=1e-4)

epochs=25

#we use cosine decay lr scheduler.
scheduler=optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=epochs)

#use GradScaler to prevent underflow
scaler=GradScaler(enabled=(device.type == "cuda"))

We initialize empty lists for metrics and also calculate weights for each task inorder to perform weighted loss calculation.

In [ ]:
best_val_acc=0.0
w_style,w_genre,w_artist=num_style**-0.5,num_genre**-0.5,num_artists**-0.5
total_w= w_style+ w_genre+ w_artist
w_style/=total_w
w_genre/=total_w
w_artist/=total_w

#we log these metrics for visualizaton
train_losses = []
val_losses = []
style_accs = []
genre_accs = []
artist_accs = []
os.makedirs("/content/drive/MyDrive/artextract_checkpoints", exist_ok=True)

Now we run the main training and validation loop.

Kindly note that I trained the model by directly cloning my repo into colab and then running !python train.py. Due to colab GPU usage limitations, I am unable to run the training and validation loop again as it will use up ~3.5 hours of GPU usage. I am currently using colab to train the model for Task 2.

You can view the !python train.py cell below this. The outputs will look exactly the same. I hope it will be excused.

In [ ]:
#training loop
for epoch in range(epochs):
  if epoch == 3:
      for p in model.cnn_backbone.parameters():
          p.requires_grad = True
  model.train()
  total_loss=0.0

  for images,style_labels,genre_labels,artist_labels in train_loader:
      images=images.to(device)
      style_labels=style_labels.to(device)
      genre_labels=genre_labels.to(device)
      artist_labels=artist_labels.to(device)
      optimizer.zero_grad(set_to_none=True)

      with autocast(device_type=device.type):
          style_out,genre_out,artist_out=model(images)
          style_loss= style_loss_fn(style_out, style_labels)
          genre_loss= genre_loss_fn(genre_out, genre_labels)
          artist_loss= artist_loss_fn(artist_out, artist_labels)

          #final loss will be weighted sum of all three losses
          loss= w_style*style_loss+ w_genre*genre_loss+ w_artist*artist_loss

      scaler.scale(loss).backward()
      scaler.unscale_(optimizer)
      nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
      scaler.step(optimizer)
      scaler.update()

      total_loss += loss.item()
  scheduler.step()

  avg_loss= total_loss / len(train_loader)
  train_losses.append(avg_loss)
  lr=optimizer.param_groups[0]['lr']
  print(f"Epoch [{epoch+1}/{epochs}], Training Loss: {avg_loss:.4f}, LR:{lr:.6f}")

  #validation loop

  model.eval()
  validation_loss=0.0
  correct_style=0
  correct_genre=0
  correct_artist=0
  total_samples=0

  all_style_preds=[]
  all_style_labels=[]
  all_genre_preds=[]
  all_genre_labels=[]
  all_artist_preds=[]
  all_artist_labels=[]
  #follows almost same logic as training loop but we dont backprop on validation data.
  with torch.no_grad():
      for images,style_labels,genre_labels,artist_labels in val_loader:
          images=images.to(device)
          style_labels=style_labels.to(device)
          genre_labels=genre_labels.to(device)
          artist_labels=artist_labels.to(device)

          with autocast(device_type=device.type):
              style_out,genre_out,artist_out=model(images)

              style_loss= style_loss_fn(style_out, style_labels)
              genre_loss= genre_loss_fn(genre_out, genre_labels)
              artist_loss= artist_loss_fn(artist_out, artist_labels)

              loss= w_style*style_loss+w_genre*genre_loss+ w_artist*artist_loss
          validation_loss+=loss.item()

          _, predicted_style = torch.max(style_out, 1)
          _, predicted_genre = torch.max(genre_out, 1)
          _, predicted_artist = torch.max(artist_out, 1)

          all_style_preds.extend(predicted_style.cpu().numpy())
          all_style_labels.extend(style_labels.cpu().numpy())
          all_genre_preds.extend(predicted_genre.cpu().numpy())
          all_genre_labels.extend(genre_labels.cpu().numpy())
          all_artist_preds.extend(predicted_artist.cpu().numpy())
          all_artist_labels.extend(artist_labels.cpu().numpy())

          correct_style += (predicted_style == style_labels).sum().item()
          correct_genre += (predicted_genre == genre_labels).sum().item()
          correct_artist += (predicted_artist == artist_labels).sum().item()
          total_samples += style_labels.size(0)

      avg_val_loss=validation_loss/len(val_loader)
      style_acc= correct_style/total_samples
      genre_acc= correct_genre/total_samples
      artist_acc= correct_artist/total_samples

      val_losses.append(avg_val_loss)
      style_accs.append(style_acc)
      genre_accs.append(genre_acc)
      artist_accs.append(artist_acc)

      combined_accuracy=(w_style*style_acc+ w_genre*genre_acc+ w_artist*artist_acc)
      #we save the best model
      if combined_accuracy > best_val_acc:
          best_val_acc= combined_accuracy
          torch.save({
          "model": model.state_dict(),
          "optimizer": optimizer.state_dict(),
          "scheduler": scheduler.state_dict(),
          "epoch": epoch,
          "best_val_acc": best_val_acc}, "/content/drive/MyDrive/artextract_checkpoints/best_model.pth")
          print(f"New best model obtained with combined accuracy: {best_val_acc:.4f}")

      style_f1 = f1_score(all_style_labels, all_style_preds, average='macro')
      genre_f1 = f1_score(all_genre_labels, all_genre_preds, average='macro')
      artist_f1 = f1_score(all_artist_labels, all_artist_preds, average='macro')


      style_class_acc = per_class_accuracy(all_style_preds, all_style_labels, num_style)
      genre_class_acc = per_class_accuracy(all_genre_preds, all_genre_labels, num_genre)
      artist_class_acc = per_class_accuracy(all_artist_preds, all_artist_labels, num_artists)

      print(f"Epoch [{epoch+1}/{epochs}] | "
          f"Validation Loss: {avg_val_loss:.4f} | "
          f"Style Acc: {style_acc:.4f} | "
          f"Genre Acc: {genre_acc:.4f} | "
          f"Artist Acc: {artist_acc:.4f} | "
          f"Style F1: {style_f1:.4f} | "
          f"Genre F1: {genre_f1:.4f} | "
          f"Artist F1: {artist_f1:.4f} | "
          )

      if epoch % 5 == 0:
          print("Style Per-Class Acc:", style_class_acc)
          print("Genre Per-Class Acc:", genre_class_acc)
          print("Artist Per-Class Acc:", artist_class_acc)

In [ ]:
!python train.py

Using device:cuda
/usr/local/lib/python3.12/dist-packages/torchvision/transforms/v2/_deprecated.py:42: UserWarning: The transform `ToTensor()` is deprecated and will be removed in a future release. Instead, please use `v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])`.Output is equivalent up to float precision.
  warnings.warn(
No of training samples: 11274
No of validation samples: 4707
/content/artextract-gsoc/Task 1 - Classification/codes/dataloader.py:57: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  style_counts = torch.bincount(torch.tensor(style_labels))
/content/artextract-gsoc/Task 1 - Classification/codes/dataloader.py:58: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTen

### Evaluation
The WikiArt subset I created (by taking inner join) has extreme class imbalance i.e. some classes have thousands of samples while others have very few. Due to this, we cannot rely on a single metric to evaluate the model. Therefore I have used 3 metrics to obtain a more reliable evaluation of the model perfomance

**Accuracy**

- Measures the overall proportion of correct predictions.

- Provides a quick and intuitive measure of the model’s overall performance.

- Useful for comparing performance across the three tasks (style, genre, artist).

- However, accuracy alone can be misleading in imbalanced datasets because a model may achieve high accuracy by performing well only on dominant classes.

**Macro F1 Score**

This is the most important and primary evaluation metric.

Precision: Fraction of predicted positive samples that are actually correct.

Recall: Fraction of actual positive samples that the model correctly identifies.

$F1 = 2 \times \frac{Precision \times Recall}{Precision + Recall}$

- Combines precision and recall, capturing both false positives and false negatives.

- Computed independently for each class and averaged across classes.

- Ensures that all classes contribute equally to the final score, including minority classes.

**Per-Class Accuracy**

This was introduced later on inorder to make sure that the model is not just focusing on dominant categories.

- Measures accuracy separately for each class.

- Helps identify classes the model struggles with or classes that are systematically confused.




Accuracy provides an overall summary of the perfomance

Macro F1 ensures that minority classes are also properly evaluated.

Per-class accuracy ensures that model is not just focusing on dominant classes.

Together, these metrics provide a balanced evaluation, ensuring that both overall performance and performance on rare categories are properly evaluated.

In [ ]:
#plot the metrics

#loss curve
epochs_range= range(1, epochs+1)
plt.figure()
plt.plot(epochs_range,train_losses, label="Training Loss")
plt.plot(epochs_range, val_losses, label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.show()

#accuracy curves
plt.figure()
plt.plot(epochs_range,style_accs, label="Style Accuracy")
plt.plot(epochs_range,genre_accs, label="Genre Accuracy")
plt.plot(epochs_range,artist_accs, label="Artist Accuracy")

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Validation Accuracy per Task")
plt.legend()
plt.show()


### Inference
Now we can use the trained model to predict the artist, style and genre of a painting of our choice.

To do that first upload your painting into MyDrive and replace the address in the final fn call with your image address.

In [ ]:
dataset = WikiArtSupervisedDataset(
        root_dir=root_dir,
        csv_file=train_csv,
        artist_map=artist_map,
        genre_map=genre_map,
        style_map=style_map,
        transform=None
    )

idx_to_style = dataset.idx_to_style
idx_to_genre = dataset.idx_to_genre
idx_to_artist = dataset.idx_to_artist

model=CNN_BiLSTM(num_style=num_style,num_genre=num_genre,num_artists=num_artists).to(device)
checkpoint = torch.load("/content/drive/MyDrive/artextract_results/best_model.pth", map_location=device)
model.load_state_dict(checkpoint["model"])
model.eval()

def predict(image_path):
    if not os.path.exists(image_path):
        print("Image not found:", image_path)
        return

    with Image.open(image_path) as img:
        img=img.convert("RGB")

    img=transform(img).unsqueeze(0).to(device)

    with torch.no_grad():
        with autocast(device_type=device.type):
            style_out, genre_out, artist_out = model(img)
        style_pred= torch.argmax(style_out, dim=1).item()
        genre_pred= torch.argmax(genre_out, dim=1).item()
        artist_pred= torch.argmax(artist_out, dim=1).item()

        style_prob = torch.softmax(style_out, dim=1)[0, style_pred].item()
        genre_prob = torch.softmax(genre_out, dim=1)[0, genre_pred].item()
        artist_prob = torch.softmax(artist_out, dim=1)[0, artist_pred].item()

    style_label = idx_to_style[style_pred]
    genre_label = idx_to_genre[genre_pred]
    artist_label = idx_to_artist[artist_pred]

    print(f"Predicted Style: {style_label} (Index={style_pred})(Confidence={style_prob:.4f})")
    print(f"Predicted Genre: {genre_label} (Index={genre_pred})(Confidence={genre_prob:.4f})")
    print(f"Predicted Artist: {artist_label} (Index={artist_pred})(Confidence={artist_prob:.4f})")

predict("/content/drive/MyDrive/INSERT_NAME")

### Outlier Detection
The goal is to identify samples that are inconsistent with the rest of the dataset i.e paintings that do not fit a particular artist or genre despite their assignment, corrupted images, or visually unusual samples.

The approach combines feature similarity analysis (k-nearest neighbors) with model prediction confidence. Images that are not visually similar to their class neighbors and predicted with low confidence are considered likely outliers.

You can find the detected outliers as a csv file of name detected_outliers.csv in your drive after the completion of this code.

In [ ]:
#inorder to detect outliers we will use knn + confidence score.
k= 20
outlier_percentage= 0.02

val_transform=v2.Compose([v2.Resize(300),v2.CenterCrop(300),
                              v2.ToTensor(),
                                v2.Normalize(mean=[0.485, 0.456, 0.406],
                                             std=[0.229, 0.224, 0.225])])

# dataset
dataset=WikiArtSupervisedDataset(root_dir, csv_path,artist_map,genre_map,style_map, transform=val_transform)
loader=DataLoader(dataset, batch_size=8, shuffle=False, num_workers=2)

# define model
model = CNN_BiLSTM(num_style, num_genre, num_artists)
checkpoint = torch.load("/content/drive/MyDrive/artextract_results/best_model.pth", map_location=device)
model.load_state_dict(checkpoint["model"])
model = model.to(device)
model.eval()


embeddings=[]
style_labels= []
genre_labels= []
artist_labels= []
style_confidence= []
genre_confidence= []
artist_confidence= []
paths= []


with torch.no_grad():
    for batch_idx, (images, style, genre, artist) in enumerate(tqdm(loader)):

      images = images.to(device)

      style_logits, genre_logits, artist_logits, features = model(images, return_features=True)

      embeddings.append(features.cpu().numpy())

      style_labels.extend(style.numpy())
      genre_labels.extend(genre.numpy())
      artist_labels.extend(artist.numpy())

      start = batch_idx * loader.batch_size
      end = start + images.size(0)

      batch_paths = dataset.data.iloc[start:end, 0].values
      paths.extend(batch_paths)

      style_prob = F.softmax(style_logits, dim=1)
      genre_prob = F.softmax(genre_logits, dim=1)
      artist_prob = F.softmax(artist_logits, dim=1)

      style_confidence.extend(style_prob.max(dim=1).values.cpu().numpy())
      genre_confidence.extend(genre_prob.max(dim=1).values.cpu().numpy())
      artist_confidence.extend(artist_prob.max(dim=1).values.cpu().numpy())


embeddings = np.concatenate(embeddings)
embeddings=embeddings/np.linalg.norm(embeddings,axis=1, keepdims=True)

neighbours= NearestNeighbors(n_neighbors=k+1,metric="cosine")
neighbours.fit(embeddings)

distances, indices= neighbours.kneighbors(embeddings)

def neighbor_agreement(labels):
    agreements = []
    for i in range(len(labels)):
        neighbor_idx = indices[i][1:]  # exclude itself
        neighbor_labels = labels[neighbor_idx]

        agree = np.mean(neighbor_labels == labels[i])
        agreements.append(agree)

    return np.array(agreements)


style_agree = neighbor_agreement(np.array(style_labels))
genre_agree = neighbor_agreement(np.array(genre_labels))
artist_agree = neighbor_agreement(np.array(artist_labels))


# convert to confidence arrays
style_conf =np.array(style_confidence)
genre_conf = np.array(genre_confidence)
artist_conf= np.array(artist_confidence)


# Outlier score
score = ((1-style_agree)+(1-style_conf) + (1-genre_agree)+(1-genre_conf) + (1-artist_agree)+(1-artist_conf))


# Select top outlier% outliers
n_outliers = int(len(score) * outlier_percentage)
outlier_idx = np.argsort(score)[-n_outliers:]

results = []

for i in outlier_idx:
    results.append({
        "path": paths[i],
        "style_label": style_labels[i],
        "genre_label": genre_labels[i],
        "artist_label": artist_labels[i],
        "score": score[i],
        "style_agree": style_agree[i],
        "genre_agree": genre_agree[i],
        "artist_agree": artist_agree[i],
        "style_conf": style_conf[i],
        "genre_conf": genre_conf[i],
        "artist_conf": artist_conf[i],
    })


df = pd.DataFrame(results)
df = df.sort_values("score", ascending=False)
#save detected outliers to csv file..
df.to_csv("/content/drive/MyDrive/artextract_results/detected_outliers.csv", index=False)


## Conclusion

The model showed strong perfomance across all three tasks as evident from the final evaluation metrics.

|Style Accuracy | Genre Accuracy | Artist Accuracy | Style F1 | Genre F1 | Artist F1 |
|---------------|---------------|----------------|----------|----------|-----------|
|0.7750 | 0.7306 | 0.8594 | 0.6458 | 0.7303 | 0.8495 |

---



---



- Artist prediction performs best (Accuracy: 0.859, F1: 0.849), indicating the model captures artist-specific visual patterns effectively.

- Genre prediction is stable and balanced (Accuracy: 0.731, F1: 0.730). The close match between accuracy and F1 suggests consistent performance across genre classes.

- Style prediction is the most challenging task (Accuracy: 0.775, F1: 0.646), likely due to strong class imbalance and visual similarity between styles.

**Per-Class Performance Analysis**

Style: Accuracy varies widely (0.00–1.00), indicating some styles are well learned while others remain difficult, likely due to severe class imbalance and limited samples.

Genre: Performance is more consistent (~0.56–1.00), with most genres showing strong accuracy, suggesting the model can reliably distinguish common visual themes.

Artist: Most artists achieve high accuracy (~0.70–0.98), indicating the model effectively captures artist-specific visual patterns.

With more number of epochs and better fine-tuning of parameters, I might be able to obtain better perfomance but that is not plausible due to GPU usage limitations on Colab.

Overall, the model demonstrates strong multi-task performance, achieving reliable predictions across all three attributes while handling a highly imbalanced dataset.